# Inlämningsuppgift 2: Utgående långvågig strålning (OLR)

## Introduktion

I Inlämningsuppgift 1 använde du enkla energibalansmodeller för att studera jordytans och atmosfärens temperatur vid strålningsbalans för olika parametervärden.
Här undersöker vi hur strålningsbalansen påverkas när koncentrationen av olika växthusgaser förändras i atmosfären.
Du kommer att simulera utgående långvågig strålning (OLR, outgoing longwave radiation) och analysera den momentana förändringen när du varierar växthusgaskoncentration.
I verkligheten skulle denna obalans resultera i temperaturförändringar som driver systemet mot ett nytt jämviktstillstånd.

Ett av målen med uppgiften är att uppskatta den *strålningsdrivning* (radiative forcing) som utsläpp av olika växthusgaser ger upphov till.
IPCC beräknar strålningsdrivning med avancerade klimat- och strålningstransportmodeller, se Figur 1 nedan.
I den här övningen använder vi en förenklad strålningstransportmodell.
Kan vi med denna enkla modell reproducera delar av denna figur?

<figure>
    <img src="./media/ipcc_5_ar6.png" width="800">
    <figcaption>
        <b>Figur 1: </b>
        Uppskattning av strålningsdrivning från olika klimatpåverkande faktorer för perioden 1750–2019.
        Siffrorna till höger anger bästa uppskattningar med osäkerhetsintervall (5–95%) inom hakparentes.
        Från IPCC AR6, 2021: <em>Climate Change 2021: The Physical Science Basis</em>.
    </figcaption>
</figure>

Uppgiften ger er också praktisk träning i att:

- Utföra numeriska beräkningar i Python
- Arbeta med flerdimensionella datastrukturer (matriser, eller mer generellt arrays)
- Visualisera resultat med tydliga figurer

## Avgränsningar

Simuleringarna är ytterst detaljerade när det gäller gasernas absorptionsspektra.
Men för att göra problemet hanterbart så bortser vi ifrån moln.
Vi antar även att marken agerar som en svart kropp och att atmosfären inte har någon horisontell variation (båda är goda approximationer).
Datan som du får täcker höjder upp till 20 km och representerar tropiska förhållande med förindustriella gasmängder.

## Teori

Modellen vi använder beräknar den spektrala radiansen,
$I$ (enhet: Wm<sup>-2</sup>Hz<sup>-1</sup>sr<sup>-1</sup>, där sr står för steradian),
för en given frekvens $\nu$ och zenitvinkel $\theta$.
En zenitvinkel på 0° motsvarar kortaste vägen genom atmosfären,
medan en zenitvinkel nära 90° ger en längre väg,
se Figur 2 nedan.
I allmänhet beror $I$ även på azimutvinkeln $\phi$,
men detta bortses från i modellen (se Avgränsningar).

<figure>
   <img src="./media/zenith-azimuth-schematic.svg" width=100%>
    <figcaption>
        <b>Figur 2: </b>(a) Illustration av utgående spektral radians för en viss vinkel.
        (b) Mer detaljerad bild av zenit- och azimutvinkel för utgående spektral radians.
    </figcaption>
</figure>

För att beräkna jordens spektrala emittansen,
$E_s$ (enhet: Wm<sup>-2</sup>Hz<sup>-1</sup>),
behöver vi integrera den spektrala radiansen som strålar uppåt från atmosfären,
det vill säga över en halvsfär vid toppen av atmosfären:

$$
E_s(\nu) = \int_{\phi=0}^{2\pi} \int_{\theta=0}^{\pi / 2} I(\nu, \theta, \phi)\cos(\theta)\sin(\theta)d\theta d\phi
$$

För de förhållanden som beskrivs under Avgränsningar så beror inte $I$ på azimutvinkeln $\phi$,
och uttrycket kan då förenklas som:

$$
E_s(\nu) = \int_{\phi=0}^{2\pi} d\phi \cdot \int_{\theta=0} ^{\pi / 2} I(\nu, \theta) \cos(\theta) \sin(\theta) d\theta
$$

Den första integralen kan lösas analytiskt och ger ett konstant värde, $C$.
Lös följande integral analytiskt.
Vad blir $C$?

$$
C = \int_{\phi = 0}^{2\pi} d\phi = ?
\tag{1}
$$

Den spektrala emittansen ges därmed av:

$$
E_s(\nu) = C \cdot \int_{\theta=0}^{\pi/2} I(\nu, \theta)\cos(\theta)\sin(\theta) d\theta
\tag{2}
$$

Jordens totala emittans (OLR),
$E$,
erhålls genom att integrera den spektrala emittansen över alla frekvenser:

$$
E = \int_{v=0}^\infty E_s(\nu)d\nu
\tag{3}
$$

En av dina uppgifter är att beräkna integralerna i Ekvation (2) och (3) med hjälp av numeriska metoder.

### Frekvens och vågtal
I figurer med strålning är det vanligt att frekvensen anges som “vågtal”, med enhet cm<sup>−1</sup>.
Detta vågtal är $1/\lambda$ där $\lambda$ är våglängden i cm.
Som exempel så visas Planckfunktionen som en funktion av vågtal i Figur 3.

<figure>
    <img src="./media/planck.png" width=600>
    <figcaption>
        <b>Figur 3: </b>Planck-kurvorna för två olika temperaturer som funktion av vågtal.
    </figcaption>
</figure>

## Kod och data

Funktioner för att beräkna radians finns i modulen `olr.py` i mappen `Kod`.
Datan kommer i två varianter i mappen `Data`,
där filerna heter `olr_data.npz` och `olr_data large.npz`.
Båda dessa filer innehåller följande variabler:

| | | |
|-|-|-|
| **f**    |Frekvenser [Hz]               |En vektor (endimensionell array). | 
| **wn**   |Vågtal [cm<sup>-1</sup>]      |En vektor, samma längd som f. | 
| **z**    |Höjder [m]                    |En vektor. | 
| **p**    |Atmosfäriskt tryck [Pa]       |En vektor, samma längd som z. |
| **t**    |Atmosfäriskt temperatur [K]   |En vektor, samma längd som z. |
| **vmr**  |Volymandelar [-]              |En matris med dimensioner (gas, z). Inkluderade gaser, i ordning, är:<br>H<sub>2</sub>O, CO<sub>2</sub>, O<sub>3</sub>, CH<sub>4</sub> och N<sub>2</sub>O. |
| **xsec**  |Absorptionstvärsnitt [m<sup>2</sup>] |En array med dimensioner (gas, f, z). Denna variabel kommer ni inte<br>använda direkt, den används av funktioner som beräknar radians. |

Använd först och främst `olr_data.npz`, som innehåller data för 3500 frekvenser.
Om du är nyfiken och vill se resultaten i en högre upplösning kan du byta till `olr_data_large.npz` när du är klar med uppgiften.
Den här filen innehåller data för 35000 frekvenser.

## Kodutveckling

Nedan ges praktiska instruktioner för att hjälpa dig utveckla koden för uppgiften.

### Steg 1

Importera NumPy som `np`:

In [ ]:
# Importera numpy här

Ladda sedan in datafilen `Data/olr_data.npz` med NumPy-funktionen `load` till en variabel med namnet `data`.
Läs dokumentationen för `load` om du inte vet hur den fungerar, t.ex. med `help(np.load)`.

In [ ]:
# Ladda in data till en variabel med namnet data

Med följande syntax tilldelar vi datan för frekvenserna till variablen f:

In [ ]:
f = data["f"]

Använda liknande syntax för att skapa en variabel för varje fysisk variabel i indatan, se tabellen ovan.

In [ ]:
# Skapa en variabel för varje fysisk variabel

Undersök storleken på varje variabel, exempelvis med `print(f.shape)`.
Försök förstå vad de olika dimensionerna betyder.
Till exempel, vilken dimension representerar höjd i `vmr`?

In [ ]:
# Undersök datan

### Steg 2

Importera modulen `olr` i mappen `Kod` och namnge den `olr`:

In [ ]:
import Kod.olr as olr

Med till exempel `help(olr)` kan du ta reda på vilka funktioner som finns i modulen och hur de fungerar.

In [ ]:
help(olr)

### Steg 3

Bli bekant med funktionen `olr.spectral_radiance`, t.ex. med `help(olr.spectral_radiance)`.
Hur anropar man funktionen, och vad returnerar den?

En parameter till funktionen är `za` för zenitvinkeln.
I vilken enhet förväntar sig funktionen att zenitvinkeln anges?

In [ ]:
# Bekanta dig med olr.spectral_radiance

Importera Matplotlib och plotta resultatet från `olr.spectral_radiance` för några zenitvinklar.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

<div class="alert alert-block alert-info">
    <b>Tips:</b>
    Får du <tt>RuntimeWarning</tt>-varningar och en array med <tt>nan</tt> och/eller <tt>inf</tt>?
    Kontrollera att alla parametrar till funktionen är passande och med korrekt enhet.
</div>

### Steg 4

Beräkna den spektrala emittansen $E_s$ enligt Ekvation (2),
det vill säga:

$$
E_s(\nu) = C \cdot \int_{\theta=0}^{\pi/2} I(\nu, \theta)\cos(\theta)\sin(\theta) d\theta
$$

där $C$ är värdet du fick när du integrerade Ekvation (1) analytiskt,
och $I$ ges av funktionen `olr.spectral_radiance`.

För att kunna utföra integralen behöver du beräkna [integranden](https://sv.wiktionary.org/wiki/integrand) för ett antal zenitvinklar ($\theta$).
Detta görs enklast med en loop.

<div class="alert alert-block alert-info">
<b>Tips:</b> Börja med några få zenitvinklar så går snabbare att köra koden.
    När du är nöjd med koden kan du öka antalet zenitvinklar för att få en mer korrekt beräkning.
</div>

I det här steget finns det några detaljer som är svåra att uppfatta om man inte är erfaren,
så här kommer lite extra hjälp:

1. Till att börja med vill vi beräkna $I(\nu, \theta)\cos(\theta)\sin(\theta)$ för ett antal värden på $\theta$.
   (Vilka värden?
   Titta på integralen i Ekvation (5) igen.)
3. När vi anropar `olr.spectral_radiance` får vi $I$ för ett specifikt värde av $\theta$ och alla angivna frekvenser $\nu$,
   alltså en endimensionell array,
   ett värde för varje frekvens.
   Dubbelkolla att detta stämmer.
   Det är dock inte $\nu$ vi vill integrera över,
   så det är inte den här endimensionella arrayen vi ska integrera.
5. Snabb sammanfattning:
   Vi beräknar $I\cos(\theta)\sin(\theta)$ för ett antal $\theta$,
   och varje gång får vi en endimensionell array.
6. Nu kan man göra något smart:
   vi kan spara allt resultat i en tvådimensionell matris,
   där resultatet för varje uträkning ($I\cos(\theta)\sin(\theta)$ för ett visst $\theta$)
   sparas som en rad eller kolumn i resultat-matrisen.

<div class="alert alert-block alert-info">
<b>Tips: </b>Kommer du ihåg hur vi skapade en tom tvådimensionell array i Python-introduktionen?
    <details>
        <summary><em>Tips:</em></summary>
        1. Det har med nollor att göra.
        <br>
        2. Det kan även hjälpa att titta på exempelkoden i Problem 4 i Python-introduktionen (under Programmeringsproblem)</a>.
    </details>
</div>

5. Målet är alltså att skapa en tvådimensionell matris med dimensionerna ($\theta$, $\nu$) eller ($\nu$, $\theta$)
   som innehåller alla integrand-värden.
   Du kan nu enkelt integrera över den här matrisen med `np.trapz` längs dimensionen som motsvarar $\theta$.

<div class="alert alert-block alert-info">
<b>Tips: </b>Vet du hur man integrerar över en tvådimensionell matris med <tt>trapz</tt>?
    <details>
        <summary><em>Tips:</em></summary>
        Se Problem 5 i Python-introduktionen (under Programmeringsproblem).
    </details>
</div>

6. Beräkna slutligen $E_s$ (glöm inte konstanten $C$).

Det är svårt i det här skedet att kontrollera om resultatet stämmer.
Det här är vanligt i programmering/dataanalys och något man får lära sig hantera/leva med.
För att göra det lättare för er får ni dock en liten ledtråd:
om du plottar den spektrala emittansen så borde de större värdena ha en storleksordning på 10<sup>-11</sup> Wm<sup>-2</sup>Hz<sup>-1</sup>.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 4

Vi kommer upprepa beräkningen av spektral emittans för olika förhållanden,
så implementera en funktion som utför uträkningen i [Steg 3](#Steg-3),
som du kallar `spectral_exitance`.
Det ska vara möjligt utföra beräkningen med funktionen för godtycklig temperaturprofil, vertikal sammansättning av gaserna, osv.
Fundera på vad för parametrar din funktionen behöver som input.

<div class="alert alert-block alert-info">
<b>Tips: </b> Titta på vilka argument <tt>olr.spectral_radiance</tt> tar, och anpassa din funktion därefter.
</div>

Testa funktionen med värdena från datafilen.
Kontrollera att resultatet stämmer med det du fick från föregående steg.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 5

Beräkna nu den totala emittansen genom att integrera över alla frekvenser enligt Ekvation (1):

$$
E = \int_{v=0}^\infty E_s(\nu)d\nu,
\tag{1}
$$

Notera att det teoretiska intervallet går från 0 till oändligheten,
men här integrerar vi över alla frekvenser som ges i inputdatan.

Resultatet ger dig simulerad OLR för jorden under de förhållanden som angavs i [Avgränsningar](#Avgränsningar).
Wow! 🎉

Det här är något vi kan relatera till.
Reflektera över om värdet du fick är rimligt.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 6

Implementera även en funktion för beräkningen i [Steg 5](#Steg-5) som du kallar `exitance`.
Denna funktion ska bland annat anropa `spectral_exitance` som du skapade i [Steg 4](#Steg-4) för att beräkna den spektrala emittansen.
Likt `spectral_exitance` ska den nya `exitance`-funktionen fungera för godtycklig temperaturprofil, vertikal sammansättning av gaserna, osv.
Vad för input kommer funktionen behöva?

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Steg 7
Nu ska du implementa funktioner för att "störa" temperatur- och vmr-profilen för att se hur det påverkar OLR.

<div class="alert alert-block alert-warning">
<b>Varning:</b>
    Python och NumPy är lite trixiga här eftersom de föredrar att jobba med referenser.
    Vad innebär detta?
    Jo, i många programmeringsspråk kopieras variabler till funktioner,
    och man behöver inte oroa sig för att råka modifiera originalvariabeln när man gör något i funktioner.
    Detta gäller alltså inte för Python.
    För att göra en kopia av en NumPy-array kan man använda <tt>.copy</tt>-metoden,
    t.ex. <tt>copy_of_t = t.copy()</tt>.
</div>

Börja med att definera en funktion `perturb_t` för att störa temperaturprofilen:
```python
perturb_t(t, dt)
```
där `t` är orginalprofilen och `dt` är en störning i K.

Vi vill begränsa störningen till höjder inom troposfären.
Vi kan här anta att `t` är ordnad från lägre till högre höjder och att tropopausen är där temperaturprofilen har sitt minsta värde.
Funktionen ska alltså ta en temperaturprofil `t` och ett värde `dt` och lägga till `dt` till `t` överallt i troposfären: alltså från början av arrayen t.o.m. där `t` har sitt minsta värde, men inte till elementen därefter.

<div class="alert alert-block alert-info">
<details>
    <summary><em>Om du behöver tips:</em></summary>
    1. Om du behöver färska upp minnet om hur indexering och slices av numpy arrays fungerar, gå tillbaka till introduktionen.
    <br>
    2. Undersök i dokumentationen eller via din favoritsökmotor om NumPy har en funktion för att hitta indexet för en arrays lägsta värde.
    <br>
    3. Det kan ofta underlätta att t.ex. plotta <tt>t</tt> för att visuellt se om du har gjort rätt.
    <br>
    4. Fortfarande fast? Det kan hjälpa att börja med Problem 6 i Python-introduktionen (under Programmeringsproblem).
</details>
</div>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

Skapa en liknande funktion för volymandelar:
```python
perturb_vmr(vmr, igas, dvmr, t)
```
där `vmr` är volymandelar,
`igas` anger indexet för gasen som ska störas,
och `dvmr` är en relativ störning (0.1 för 10% ökning o.s.v.).
Även `perturb_vmr` ska bara störa värdena i troposfären (och `t` måste därför vara input även till den här funktionen).

In [ ]:
# Skriv din kod här, lägg till celler efter behov

## Frågor
Svaren till följande frågor/deluppgifter ska lämnas in individuellt på Canvas.

<div class="alert  alert-block alert-info">
Kommer du ihåg hur man sparar figurer? Om inte, se avsnittet om Matplotlib i Python-introduktionen.
</div>

### Del 1

#### Fråga 1

Skapa en figur som visar spektral radians som funktion av vågtal för två olika zenitvinklar: 10° och 80°.

Märk kurvorna tydligt, t.ex. med en legend. För denna och alla kommande figurer: ange storhet och enhet på både x- och y-axeln.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 2

Plotta spektral emittans som funktion av vågtal. Glöm inte att ange enheter.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 3

Beräkna OLR (emittans) för standardvärdena i datafilen. Vilket värde får du?

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 4

Vad har OLR i föregående fråga för enhet? Ange svaret utan mellanslag (för att den automatiska rättningen ska fungera).

**Svar:** (Skriv ditt svar här om du vill)

#### Fråga 5

Hur kom du fram till att ditt beräknade OLR-värde är rimligt?

**Svar:** (Skriv ditt svar här om du vill)

### Del 2

#### Fråga 6

CO<sub>2</sub>-halten har ökat med ca 47 % mellan preindustriella nivåer och 2019.

Plotta skillnaden i spektral emittans (som funktion av vågtal) när CO<sub>2</sub>-halten i troposfären ökar med 47 %.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 7

Vad är förändringen i OLR efter CO<sub>2</sub>-ökningen? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 8

CH<sub>4</sub>-halten har ökat från ca 729 ppb år 1750 till 1866 ppb år 2019, vilket motsvarar en relativ ökning på 156%.

Plotta skillnaden i spektral emittans (som funktion av vågtal) när CH<sub>4</sub>-halten i troposfären ökar med 156%.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 9

Vad är förändringen i OLR efter CH<sub>4</sub>-ökningen? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

#### Fråga 10

Vad är den relativa ökningen i N<sub>2</sub>O-halten mellan 1750 och 2019?
Använd värden från [IPCC, 2021: Annex III](https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_AnnexIII.pdf) och ange svaret i procent (%).

**Svar:** (Skriv ditt svar här om du vill)

#### Fråga 11

Vad är förändringen i OLR efter förändringen i N<sub>2</sub>O-halten? Svara i samma enhet som i fråga 4 och var noga med tecknet.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Fråga 5
Gör en figur som visar hur spektral emittans ändras för en 35%-ig ökning av CO<sub>2</sub> i troposfären.
Vad är förändringen av OLR?
35% är ungefär så mycket CO<sub>2</sub> hade ökat ifrån sitt förindustriella värde 2019.
Hur väl stämmer ditt resultat med IPCCs värde?
(Se Figur 3 nedan.)
<figure>
    <img src="./media/ipcc_5_ar6.png" width="800">
    <figcaption>
        <b>Figur 3: </b>Uppskattning av "radiative forcing". Från IPCC AR6, 2021: <em>Climate Change 2021: The Physical Science Basis</em>.
    </figcaption>
</figure>

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Fråga 6

Som 5 men modifiera O<sub>3</sub> och jämför med IPCC.
Ökningen av O<sub>3</sub> varierar kraftigt mellan regioner, men den globala ökningen av troposfäriskt ozon är i storleksordningen 30%.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Fråga 7

Som 5 men modifiera CH<sub>4</sub> med ett procenttal som representerar ökningen 2019, och jämför med IPCC.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Fråga 8

Som 5 men modifiera N<sub>2</sub>O med ett procental som representerar ökningen 2019, och jämför med IPCC.

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Fråga 9

Ungefär hur mycket måste du ändra H<sub>2</sub>O för att få samma ändring i OLR som den 35%-iga ökningen i CO<sub>2</sub> gav?
Om du slår samman resultaten ifrån 5–9, vilken är den starkaste växthusgasen när det kommer till relativa förändringar?

In [ ]:
# Skriv din kod här, lägg till celler efter behov

### Fråga 10

Hur mycket förändras OLR vid en fördubbling av CO<sub>2</sub>?
Vilken temperaturökning behövs för att föra OLR tillbaka (d.v.s. till OLR-värdet ifrån fråga 3)?

(Tips:
Du kan hitta temperaturvärdet med "trial and error".
Men kan du även beräkna den med någon fysikalisk lag?)

In [ ]:
# Skriv din kod här, lägg till celler efter behov